# TPC-DS Benchmark Visualization: Trino vs. Apache Spark

This notebook provides paper-quality visualizations for the TPC-DS benchmark results, comparing performance across latency, memory usage, and throughput.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Robustly find project root and scripts directory
def find_project_paths():
    curr = Path(os.getcwd()).resolve()
    for _ in range(5):
        if (curr / "benchmark" / "scripts" / "common.py").exists():
            return curr, curr / "benchmark" / "scripts"
        if (curr / "scripts" / "common.py").exists():
            return curr.parent, curr / "scripts"
        curr = curr.parent
    return None, None

project_root, scripts_path = find_project_paths()
if scripts_path:
    if str(scripts_path) not in sys.path:
        sys.path.append(str(scripts_path))
    print(f"Project root: {project_root}")
    print(f"Scripts path: {scripts_path}")
else:
    print("WARNING: Could not find project structure automatically.")

try:
    from common import load_results_df, load_env
    print("Successfully imported common utilities.")
except ImportError:
    print("ERROR: Could not import common.py. Ensure it is in the scripts directory.")

# Set paper-quality plot settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 9,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'figure.titlesize': 16,
    'figure.dpi': 120,
    'savefig.bbox': 'tight',
})

# Define engine colors for consistency
COLORS = {"trino": "#3498db", "spark": "#e67e22"}
ENGINE_ORDER = ["trino", "spark"]

## 1. Data Loading & Preprocessing
We load the raw JSONL results and filter for successful, measured runs.

In [ ]:
df = load_results_df()
env = load_env()

if df.empty:
    print("!!! ERROR: No benchmark results found via load_results_df(). !!!")
    # Fallback search
    search_paths = [
        Path(os.getcwd()) / "spark_results.jsonl",
        Path(os.getcwd()) / "trino_results.jsonl",
        Path(os.getcwd()).parent / "raw" / "spark_results.jsonl",
        project_root / "benchmark" / "results" / "raw" / "spark_results.jsonl"
    ] if project_root else []
    
    records = []
    from common import load_jsonl
    for p in search_paths:
        if p.exists():
            records.extend(load_jsonl(p))
    
    if records:
        df = pd.DataFrame(records)
        df["latency_s"] = df.apply(
            lambda r: r["engine_internal_time"] if r.get("engine_internal_time", 0) > 0 else r.get("wall_time_seconds", 0),
            axis=1
        )
    else:
        print("Manual search also failed.")

if not df.empty:
    # Filter for measured runs only
    df_m = df[(df['run_type'] == 'measured') & (df['status'] == 'success')].copy()

    if df_m.empty:
        print("!!! WARNING: Found results but no successful 'measured' runs. !!!")
        df_agg = pd.DataFrame()
    else:
        # Aggregate per query
        df_agg = df_m.groupby(['engine', 'query_name']).agg({
            'latency_s': 'median',
            'peak_memory_bytes': 'max',
            'throughput_qps': 'median'
        }).reset_index()
        
        # Extract numeric ID for sorting and display
        df_agg['query_num'] = df_agg['query_name'].str.extract(r'(\d+)').astype(int)
        df_agg = df_agg.sort_values(['query_num', 'engine'])

        print(f"Loaded {len(df)} total records.")
        print(f"Analyzing {df_agg['query_name'].nunique()} unique queries.")
else:
    df_agg = pd.DataFrame()

## 2. Geometric Mean of Latency
The geometric mean is the standard way to compare benchmark suites like TPC-DS.

In [ ]:
if not df_agg.empty:
    def geometric_mean(series):
        return np.exp(np.log(series[series > 0]).mean())

    geomeans = df_agg.groupby('engine')['latency_s'].apply(geometric_mean).reindex(ENGINE_ORDER)

    plt.figure(figsize=(10, 6))
    # Assigning hue=geomeans.index and legend=False to fix FutureWarning
    ax = sns.barplot(x=geomeans.index, y=geomeans.values, hue=geomeans.index, 
                     palette=[COLORS[e] for e in geomeans.index], legend=False)
    plt.ylabel("Geometric Mean Latency (s)")
    plt.xlabel("Engine")
    plt.title("Overall Performance: Geometric Mean Latency")

    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}s', (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='center', fontsize=12, color='black', xytext=(0, 5), 
                    textcoords='offset points')

    plt.tight_layout()
    plt.show()
else:
    print("No data to plot geomean.")

## 3. Per-Query Latency Comparison
Visualizing individual query performance by query number.

In [ ]:
if not df_agg.empty:
    plt.figure(figsize=(24, 8))
    sns.barplot(data=df_agg, x='query_num', y='latency_s', hue='engine', 
                hue_order=ENGINE_ORDER, palette=COLORS)

    plt.ylabel("Median Latency (seconds)")
    plt.xlabel("Query Number")
    plt.xticks(rotation=90)
    plt.title("TPC-DS Query Latency: Trino vs. Spark")
    plt.legend(title="Engine")
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot latency comparison.")

### Log-Scale Latency
Useful when latency varies significantly across queries.

In [ ]:
if not df_agg.empty:
    plt.figure(figsize=(24, 8))
    g = sns.barplot(data=df_agg, x='query_num', y='latency_s', hue='engine', 
                    hue_order=ENGINE_ORDER, palette=COLORS)
    g.set_yscale("log")

    plt.ylabel("Latency (s) - Log Scale")
    plt.xlabel("Query Number")
    plt.xticks(rotation=90)
    plt.title("TPC-DS Query Latency (Log Scale)")
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot log-scale latency.")

## 4. Peak Memory Consumption (Max RSS)
Comparing peak memory usage per query.

In [ ]:
if not df_agg.empty:
    df_agg['peak_mem_gb'] = df_agg['peak_memory_bytes'] / (1024**3)

    plt.figure(figsize=(24, 8))
    sns.barplot(data=df_agg, x='query_num', y='peak_mem_gb', hue='engine', 
                hue_order=ENGINE_ORDER, palette=COLORS)

    plt.ylabel("Peak Memory (GiB)")
    plt.xlabel("Query Number")
    plt.xticks(rotation=90)
    plt.title("Peak Memory Usage (Max RSS) per Query")
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot memory profile.")

## 5. Speedup Analysis
Spark/Trino ratio. Ratio > 1 means Trino is faster.

In [ ]:
if not df_agg.empty:
    pivoted = df_agg.pivot(index='query_num', columns='engine', values='latency_s')
    if 'spark' in pivoted.columns and 'trino' in pivoted.columns:
        pivoted['speedup'] = pivoted['spark'] / pivoted['trino']
        pivoted = pivoted.dropna()

        if not pivoted.empty:
            plt.figure(figsize=(24, 8))
            colors_speedup = ['#2ecc71' if x > 1 else '#e74c3c' for x in pivoted['speedup']]
            # Fix FutureWarning by assigning hue and setting legend=False
            sns.barplot(x=pivoted.index, y=pivoted['speedup'], hue=pivoted.index, 
                        palette=colors_speedup, legend=False)
            plt.axhline(1, color='black', linestyle='--')
            plt.ylabel("Spark / Trino Speedup Ratio")
            plt.xlabel("Query Number")
            plt.xticks(rotation=90)
            plt.title("Relative Performance: Trino Speedup over Spark (Ratio > 1 is faster for Trino)")
            plt.tight_layout()
            plt.show()
        else:
            print("No overlapping successful queries between engines.")
    else:
        print("Missing data for one of the engines.")
else:
    print("No data for speedup analysis.")

## 6. Latency vs. Memory Scatter Plot
Trade-off between execution time and resource consumption.

In [ ]:
if not df_agg.empty:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_agg, x='latency_s', y='peak_mem_gb', hue='engine', 
                    style='engine', s=100, palette=COLORS)

    plt.xscale('log')
    plt.xlabel("Median Latency (s) - Log Scale")
    plt.ylabel("Peak Memory (GiB)")
    plt.title("Latency vs. Memory Trade-off")
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot trade-off scatter plot.")